# Circuit Modularity Scores

For each universal object, how "modular" is its circuit — how distinct is it from the rest of the population?

**Metric:** For each object at each layer, compute its mean Jaccard similarity to all other objects. Low similarity = distinctive circuit (the object has neurons that are specific to it). High similarity = shared/generic circuit (the object's neurons overlap with many others).

**Modularity score:** At each layer, test whether the object's mean-Jaccard-to-others is significantly *lower* than what a random object would have (permutation test, p < 0.05). Count the number of layers where the object passes → score out of N_LAYERS.

- Score N/N = maximally modular — distinct at every layer
- Score 0/N = not modular at all — indistinguishable from a random circuit

In [ ]:
# Cell 1 – Setup & load
import subprocess, sys, os, shutil
for pkg in ["h5py", "seaborn", "matplotlib", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)

if IN_COLAB:
    DATA_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas"
else:
    DATA_DIR = "/Users/piotrwilam/Data/CSP-Atlas"

LOCAL_SRC = "/Users/piotrwilam/Code/CSP-Atlas/src"
COLAB_SRC = "/content/drive/MyDrive/CODE/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import matplotlib.colors as mcolors
from module2.io_utils import load_atlas_hdf5

# Configuration — single universal file
UNIVERSAL_FILE = "universal_106x50.h5"
UNIVERSAL_HDF5 = f"{DATA_DIR}/{UNIVERSAL_FILE}"

print(f"Loading: {UNIVERSAL_HDF5}")
atlas = load_atlas_hdf5(UNIVERSAL_HDF5)
print(f"Loaded — {len(atlas['pair_masks'])} pairs")

In [ ]:
# Cell 2 – Build mask data + utilities
def build_mask_data(atlas):
    um, pm = atlas["universal_masks"], atlas["pair_masks"]
    lids = sorted({lid for lm in pm.values() for lid in lm})
    cnames, rows = [], []
    for name in sorted(um["ast"]):
        vec = [um["ast"][name].get(lid, np.zeros(2048)).astype(np.float32) for lid in lids]
        cnames.append(f"AST:{name}"); rows.append(np.concatenate(vec))
    for name in sorted(um["builtin"]):
        vec = [um["builtin"][name].get(lid, np.zeros(2048)).astype(np.float32) for lid in lids]
        cnames.append(f"BLT:{name}"); rows.append(np.concatenate(vec))
    mm = np.array(rows)
    return mm, cnames, lids, len(cnames), mm.shape[1] // len(lids)

def get_selective_layer(mm, li, npl, nc):
    s, e = li * npl, (li + 1) * npl
    ls = mm[:, s:e]
    cs = ls.sum(axis=0)
    return ls[:, (cs > 0) & (cs < nc)]

def mean_jaccard_to_others(layer_sel, idx, all_indices):
    """Mean Jaccard of one object to all others."""
    if layer_sel.shape[1] == 0:
        return 0.0
    a = layer_sel[idx].astype(bool)
    vals = []
    for j in all_indices:
        if j == idx:
            continue
        b = layer_sel[j].astype(bool)
        inter = (a & b).sum(); union = (a | b).sum()
        vals.append(inter / union if union > 0 else 0.0)
    return np.mean(vals) if vals else 0.0

mm, circuit_names, layer_ids, n_circuits, neurons_per_layer = build_mask_data(atlas)
all_indices = list(range(n_circuits))
n_layers = len(layer_ids)
print(f"Circuits: {n_circuits} | Layers: {n_layers} | Neurons/layer: {neurons_per_layer}")

In [ ]:
# Cell 3 – Compute modularity: mean-J-to-others + permutation test at each layer
from tqdm.auto import tqdm

N_PERMS = 500

print(f"Computing modularity scores for {n_circuits} objects across {n_layers} layers...")

results = []
for li, lid in enumerate(tqdm(layer_ids, desc="Layers")):
    ls = get_selective_layer(mm, li, neurons_per_layer, n_circuits)
    n_sel = ls.shape[1]

    if n_sel == 0:
        for idx in range(n_circuits):
            results.append({"circuit": circuit_names[idx],
                            "layer": lid, "mean_J_to_others": 0.0,
                            "p_value": 1.0, "distinctive": False, "n_selective": 0})
        continue

    # Compute mean-J-to-others for every object
    obj_jaccards = []
    for idx in range(n_circuits):
        mj = mean_jaccard_to_others(ls, idx, all_indices)
        obj_jaccards.append(mj)

    # Random baseline: sample N_PERMS random objects, get their mean-J-to-others
    np.random.seed(42)
    random_jaccards = np.array([obj_jaccards[np.random.randint(n_circuits)] for _ in range(N_PERMS)])

    # For each object: is its mean-J significantly LOWER than random?
    for idx in range(n_circuits):
        mj = obj_jaccards[idx]
        p_low = (random_jaccards <= mj).mean()
        results.append({
            "circuit": circuit_names[idx], "layer": lid,
            "mean_J_to_others": mj, "p_value": p_low,
            "distinctive": p_low < 0.05, "n_selective": n_sel,
        })

results_df = pd.DataFrame(results)
print(f"\nComputed {len(results_df)} scores ({n_circuits} objects × {n_layers} layers)")

In [ ]:
# Cell 4 – Modularity score table (ranked)
scores = results_df.groupby("circuit").agg(
    modularity_score=("distinctive", "sum"),
    mean_J_overall=("mean_J_to_others", "mean"),
).reset_index()

# Add type (AST vs BLT)
scores["type"] = scores["circuit"].apply(lambda x: "AST" if x.startswith("AST:") else "BLT")
scores = scores.sort_values("modularity_score", ascending=False).reset_index(drop=True)

print(f"Modularity scores for {len(scores)} circuits (max possible: {n_layers})")
print(f"Mean score: {scores['modularity_score'].mean():.1f}")
print(f"Circuits with score > 0: {(scores['modularity_score'] > 0).sum()}")
print(f"Circuits with score = 0: {(scores['modularity_score'] == 0).sum()}")
print()
display(scores)

In [ ]:
# Cell 5 – Score distribution + AST vs BLT comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ast_scores = scores[scores["type"] == "AST"]["modularity_score"]
blt_scores = scores[scores["type"] == "BLT"]["modularity_score"]

# Histogram of scores
ax = axes[0]
ax.hist(scores["modularity_score"], bins=range(0, n_layers + 2), edgecolor="black", alpha=0.7, color="#2c7bb6")
ax.set_xlabel(f"Modularity score (out of {n_layers})")
ax.set_ylabel("# circuits")
ax.set_title("Distribution of modularity scores")
ax.axvline(scores["modularity_score"].mean(), color="red", linestyle="--",
           label=f"Mean = {scores['modularity_score'].mean():.1f}")
ax.legend()

# AST vs BLT
ax = axes[1]
ax.hist(ast_scores, bins=range(0, n_layers + 2), alpha=0.6, color="#2c7bb6", label=f"AST (n={len(ast_scores)})", edgecolor="black")
ax.hist(blt_scores, bins=range(0, n_layers + 2), alpha=0.6, color="#fdae61", label=f"BLT (n={len(blt_scores)})", edgecolor="black")
ax.set_xlabel("Modularity score")
ax.set_ylabel("# circuits")
ax.set_title("AST vs Builtin modularity")
ax.legend()

# Score vs mean-J-to-others
ax = axes[2]
ax.scatter(scores["mean_J_overall"], scores["modularity_score"], alpha=0.5, s=30,
           c=scores["type"].map({"AST": "#2c7bb6", "BLT": "#fdae61"}))
ax.set_xlabel("Mean Jaccard to all others (avg across layers)")
ax.set_ylabel("Modularity score")
ax.set_title("Lower similarity → higher modularity")
ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\nAST circuits: mean score = {ast_scores.mean():.1f}, median = {ast_scores.median():.0f}")
print(f"BLT circuits: mean score = {blt_scores.mean():.1f}, median = {blt_scores.median():.0f}")

In [ ]:
# Cell 6 – Heatmap: all circuits × layers
ref = results_df.copy()
ref_lids = sorted(ref["layer"].unique())

pivot_j = ref.pivot(index="circuit", columns="layer", values="mean_J_to_others")
ordered = scores.sort_values("modularity_score", ascending=False)["circuit"].tolist()
pivot_j = pivot_j.reindex(ordered)

pivot_sig = ref.pivot(index="circuit", columns="layer", values="distinctive").reindex(ordered)

fig, ax = plt.subplots(figsize=(10, max(8, len(ordered) * 0.16)))

_colors_5 = ["#d7191c", "#fdae61", "#ffffbf", "#abd9e9", "#2c7bb6"]
_cmap_5   = mcolors.ListedColormap(_colors_5)
_bounds_5 = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
_norm_5   = mcolors.BoundaryNorm(_bounds_5, _cmap_5.N)

sns.heatmap(pivot_j.values, ax=ax, vmin=0, vmax=1, cmap=_cmap_5, norm=_norm_5,
            xticklabels=ref_lids, yticklabels=ordered, cbar_kws={"label": "Mean J to others"})
ax.set_title("Mean Jaccard to all others per layer\n(blue = distinctive, red = shared)")
ax.set_xlabel("Layer")
ax.tick_params(axis="y", labelsize=6)
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 – Top 10 most modular + top 10 least modular
print(f"TOP 10 MOST MODULAR (distinctive circuits):")
print("=" * 70)
top = scores.head(10)
for _, row in top.iterrows():
    print(f"  {row['modularity_score']:>2}/{n_layers} | J={row['mean_J_overall']:.3f} | {row['circuit']}")

print(f"\nTOP 10 LEAST MODULAR (shared/generic circuits):")
print("=" * 70)
bottom = scores.tail(10).iloc[::-1]
for _, row in bottom.iterrows():
    print(f"  {row['modularity_score']:>2}/{n_layers} | J={row['mean_J_overall']:.3f} | {row['circuit']}")

In [ ]:
# Cell 8 – Export report
import datetime, json

date_str = datetime.date.today().isoformat()

# Save TXT report
txt_path = os.path.join(DATA_DIR, f"report_4_modularity_{date_str}.txt")
with open(txt_path, "w") as f:
    f.write("CIRCUIT MODULARITY SCORES — REPORT\n")
    f.write(f"Generated: {datetime.datetime.now().isoformat()}\n")
    f.write(f"Input: {UNIVERSAL_FILE}\n")
    f.write(f"Permutations: {N_PERMS} | Layers: {n_layers}\n")
    f.write(f"Circuits: {n_circuits} | Max score: {n_layers}\n")
    f.write(f"\n{'='*80}\n")

    f.write(f"\nSUMMARY\n")
    f.write(f"Mean score     : {scores['modularity_score'].mean():.1f}/{n_layers}\n")
    f.write(f"Score > 0      : {(scores['modularity_score'] > 0).sum()}/{n_circuits}\n")
    f.write(f"Score = 0      : {(scores['modularity_score'] == 0).sum()}/{n_circuits}\n")
    f.write(f"AST mean score : {ast_scores.mean():.1f}\n")
    f.write(f"BLT mean score : {blt_scores.mean():.1f}\n")

    f.write(f"\n{'='*80}\n")
    f.write("FULL RANKED TABLE\n")
    f.write(f"{'='*80}\n\n")
    f.write(scores.to_string(index=False))
    f.write("\n")

    f.write(f"\n{'='*80}\n")
    f.write("TOP 10 MOST MODULAR\n")
    for _, row in scores.head(10).iterrows():
        f.write(f"  {row['modularity_score']:>2}/{n_layers} | J={row['mean_J_overall']:.3f} | {row['circuit']}\n")

    f.write(f"\nTOP 10 LEAST MODULAR\n")
    for _, row in scores.tail(10).iloc[::-1].iterrows():
        f.write(f"  {row['modularity_score']:>2}/{n_layers} | J={row['mean_J_overall']:.3f} | {row['circuit']}\n")

# Save JSON report
json_path = os.path.join(DATA_DIR, f"report_4_modularity_{date_str}.json")
report = {
    "generated_at": datetime.datetime.now().isoformat(),
    "input_file": UNIVERSAL_FILE,
    "n_circuits": n_circuits,
    "n_layers": n_layers,
    "n_perms": N_PERMS,
    "mean_score": float(scores["modularity_score"].mean()),
    "score_gt_0": int((scores["modularity_score"] > 0).sum()),
    "score_eq_0": int((scores["modularity_score"] == 0).sum()),
    "ast_mean_score": float(ast_scores.mean()),
    "blt_mean_score": float(blt_scores.mean()),
    "scores": scores.to_dict(orient="records"),
}
with open(json_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Reports saved:")
print(f"  TXT:  {txt_path}")
print(f"  JSON: {json_path}")